BELOW FISCAL YEAR STARTING ON 01/10

In [ ]:
import pandas as pd
import numpy as np

# -------- CONFIG --------
INFILE  = "/Users/jon/Desktop/Ironhack/Unit 7 - Final Project/Final-Project-Ironhack/Type of analysis/Analisis temporal/Merged files/csv/apple/merged_apple_financials_2009_2025.csv"     # <-- change if needed
OUTFILE = "merged_apple_financials_clean_09_25_missing.csv"

FLOW_COLS    = ["revenue", "net_income"]                  # period flows
INSTANT_COLS = ["total_assets", "total_liabilities", "shareholders_equity"]  # end-of-period

# -------- LOAD --------
financials_09_25 = pd.read_csv(INFILE, parse_dates=["filing_date", "period_end"])
for c in FLOW_COLS + INSTANT_COLS:
    if c in financials_09_25.columns:
        financials_09_25[c] = pd.to_numeric(financials_09_25[c], errors="coerce")

# -------- HELPERS --------
def is_valid_period_end(d: pd.Timestamp) -> bool:
    """
    Apple uses a 52/53-week calendar; quarter ends are Saturdays near:
      - Q1: last Sat of Dec
      - Q2: last Sat of Mar OR first Sat of Apr
      - Q3: last Sat of Jun OR first Sat of Jul
      - Q4: last Sat of Sep OR first Sat of Oct (rare 53-week)
    """
    if pd.isna(d): return False
    if d.weekday() != 5:  # Saturday
        return False
    m, day = d.month, d.day
    if m == 12 and 24 <= day <= 31: return True
    if (m == 3 and day >= 24) or (m == 4 and day <= 7): return True
    if (m == 6 and day >= 24) or (m == 7 and day <= 7): return True
    if (m == 9 and day >= 24) or (m == 10 and day <= 7): return True
    return False

def fiscal_year_from_period_end(d: pd.Timestamp) -> int:
    # Dec belongs to next FY; others use same year
    if d.month == 12:
        return d.year + 1
    return d.year

def quarter_label_from_period_end(d: pd.Timestamp) -> str:
    # Apple fiscal: Q1 ends in late Dec → FY+1; Q2 late Mar/early Apr; Q3 late Jun/early Jul; Q4 late Sep/early Oct
    if d.month == 12: return f"Q1 {d.year + 1}"
    if d.month in (3, 4): return f"Q2 {d.year}"
    if d.month in (6, 7): return f"Q3 {d.year}"
    if d.month in (9, 10): return f"Q4 {d.year}"
    return None

def fq_num_from_label(q: str):
    if isinstance(q, str) and q.startswith("Q"):
        try: return float(q[1])
        except: return np.nan
    return np.nan

# -------- 1) DEDUP: keep latest filing per (form, period_end) --------
financials_09_25 = (
    financials_09_25
    .sort_values(["period_end", "form", "filing_date"])     # earliest .. latest
    .groupby(["period_end", "form"], as_index=False)
    .last()                                                 # keep latest filing
)

# -------- 2) DROP obvious spurious period_end --------
financials_09_25 = financials_09_25[financials_09_25["period_end"].apply(is_valid_period_end)].copy()

# Also: keep only September/early-Oct 10-Ks (true annuals)
financials_09_25 = financials_09_25[~((financials_09_25["form"] == "10-K") &
                                      ~(financials_09_25["period_end"].dt.month.isin([9, 10])) )].copy()

# -------- 3) ASSIGN fiscal labels --------
financials_09_25["fy"]      = financials_09_25["period_end"].apply(fiscal_year_from_period_end).astype("Int64")
financials_09_25["quarter"] = financials_09_25["period_end"].apply(quarter_label_from_period_end)
financials_09_25["fq"]      = financials_09_25["quarter"].map(fq_num_from_label)

# -------- 4) SYNTHESIZE Q4 if missing (flows from 10-K - (Q1+Q2+Q3); instants from 10-K) --------
q = financials_09_25[financials_09_25["form"] == "10-Q"].copy()
k = financials_09_25[financials_09_25["form"] == "10-K"].copy()

# Sum flows for Q1..Q3 per FY
sum_q1_3 = (q[q["fq"].isin([1.0, 2.0, 3.0])]
            .groupby("fy")[FLOW_COLS]
            .sum(min_count=1))

# Annual flows & instants per FY from 10-K (one per FY after dedup)
annual = (k.sort_values("period_end")
            .drop_duplicates(subset=["fy"], keep="last")
            .set_index("fy"))

annual_flows    = annual[FLOW_COLS] if set(FLOW_COLS).issubset(annual.columns) else pd.DataFrame(index=annual.index)
annual_instants = annual[INSTANT_COLS] if set(INSTANT_COLS).issubset(annual.columns) else pd.DataFrame(index=annual.index)
annual_meta     = annual[["filing_date", "period_end"]] if {"filing_date","period_end"}.issubset(annual.columns) else pd.DataFrame(index=annual.index)

# Compute Q4 flows
q4_flows = (annual_flows - sum_q1_3).dropna(how="all")

# Which FYs still lack a Q4 10-Q?
have_q4 = set(q[q["fq"] == 4.0]["fy"].dropna().astype(int).tolist())
need_q4 = [fy for fy in q4_flows.index if fy not in have_q4]

q4_rows = []
for fy in need_q4:
    p_end  = annual_meta.loc[fy, "period_end"] if "period_end" in annual_meta.columns else pd.Timestamp(f"{fy}-09-30")
    fdate  = annual_meta.loc[fy, "filing_date"] if "filing_date" in annual_meta.columns else p_end

    row = {
        "filing_date": fdate,
        "period_end":  p_end,
        "form":        "10-Q",
        "fy":          int(fy),
        "quarter":     f"Q4 {fy}",
        "fq":          4.0,
        "synthetic":   True,
        "note":        "Q4 flows = 10K - (Q1+Q2+Q3); instants from 10K",
    }
    for c in FLOW_COLS:
        row[c] = q4_flows.loc[fy, c] if c in q4_flows.columns else np.nan
    for c in INSTANT_COLS:
        row[c] = annual_instants.loc[fy, c] if c in annual_instants.columns else np.nan
    q4_rows.append(row)

q4_df = pd.DataFrame(q4_rows)

# Mark originals
financials_09_25["synthetic"] = False
financials_09_25["note"] = ""

# Combine and sort
full = pd.concat([financials_09_25, q4_df], ignore_index=True)
full["filing_date"] = pd.to_datetime(full["filing_date"])
full = full.sort_values(["fy", "period_end", "form", "filing_date"]).reset_index(drop=True)

# === 5) AÑADIR FILAS EN BLANCO SI FALTAN TRIMESTRES (Q1..Q4) POR FY ===
# Trabajamos sobre vista trimestral (10-Q o sintéticos) para evitar duplicados por 10-K
quarterly_view = full[(full["form"] == "10-Q") | (full["synthetic"])].copy()

# Extrae 'FY' y el código de trimestre (Q1..Q4) a partir de 'quarter'
quarterly_view["fy"] = quarterly_view["quarter"].str.extract(r"\b(\d{4})\b").astype("Int64")
quarterly_view["qcode"] = quarterly_view["quarter"].str.extract(r"(Q[1-4])")

# Si hay más de una fila por (fy, q) (p.ej., múltiples filings),
# prioriza 10-Q frente a 10-K (por si se coló alguno), y la última filing_date
priority = {"10-Q": 0, "10-K": 1}
quarterly_view["form_priority"] = quarterly_view["form"].map(priority).fillna(2).astype(int)

quarterly_view = (
    quarterly_view.sort_values(["fy", "qcode", "form_priority", "filing_date"])
                  .groupby(["fy", "qcode"], as_index=False)
                  .last()
)

# Construye el MultiIndex completo FY x (Q1..Q4)
fy_min, fy_max = int(quarterly_view["fy"].min()), int(quarterly_view["fy"].max())
full_index = pd.MultiIndex.from_product(
    [range(fy_min, fy_max + 1), ["Q1", "Q2", "Q3", "Q4"]],
    names=["fy", "qcode"]
)

# Reindexa para introducir filas en blanco donde falten trimestres
quarterly_full = (
    quarterly_view.set_index(["fy", "qcode"])
                  .reindex(full_index)
                  .reset_index()
)

# Reconstruye la etiqueta 'quarter' (p.ej., "Q1 2014")
quarterly_full["quarter"] = quarterly_full["qcode"] + " " + quarterly_full["fy"].astype("Int64").astype(str)

# Integra de vuelta con el DataFrame 'full':
#  - Eliminamos posibles duplicados por 'quarter' en 'full'
#  - Usamos 'quarterly_full' como base trimestral definitiva (con NaNs donde faltaba)
cols_to_keep = full.columns.tolist()
# Asegura que 'quarterly_full' tenga todas las columnas (las ausentes se crean como NaN)
for c in cols_to_keep:
    if c not in quarterly_full.columns:
        quarterly_full[c] = np.nan

# Nos quedamos con las columnas en el mismo orden
full = quarterly_full[cols_to_keep]

# (Optional) Keep purely quarterly view (drop 10-Ks) once Q4 is synthesized:
# full = full[(full["form"] == "10-Q") | (full["synthetic"])].copy()

# Quick check: quarters present per FY
check = (full.assign(fq_lbl=lambda d: d["quarter"].str.extract(r"(Q[1-4])"))
              .pivot_table(index="fy", columns="fq_lbl", values="period_end", aggfunc="count")
              .fillna(0).astype(int))
print("Quarters present per FY:\n", check)

# Drop unecessary columns

full = full.drop(columns=["filing_date", "fy", "fq"])

# Save
full.to_csv(OUTFILE, index=False)
print(f"[DONE] Saved cleaned dataset → {OUTFILE}")


Quarters present per FY:
 fq_lbl  Q1  Q2  Q3  Q4
fy                    
2012     1   1   0   0
2013     0   0   0   0
2014     0   0   0   0
2015     0   0   0   0
2016     0   0   0   0
2017     0   0   0   0
2018     0   0   0   0
2019     0   0   0   0
2020     0   0   0   0
2021     0   0   0   0
2022     0   0   0   0
2023     1   0   0   1
[DONE] Saved cleaned dataset → merged_microsoft_financials_clean_09_25_missing.csv


CODE BELOW FOR FISCAL YEAR STARTING ON 01/07

In [2]:
import pandas as pd
import numpy as np

# -------- CONFIG --------
INFILE  = "/Users/jon/Desktop/Ironhack/Unit 7 - Final Project/Final-Project-Ironhack/Type of analysis/Analisis temporal/Merged files/csv/microsoft/merged_microsoft_financials_2009_2025.csv"  # <-- MSFT file
OUTFILE = "merged_microsoft_financials_clean_09_25_missing.csv"

FLOW_COLS    = ["revenue", "net_income"]                  # period flows
INSTANT_COLS = ["total_assets", "total_liabilities", "shareholders_equity"]  # end-of-period

# -------- LOAD --------
financials_09_25 = pd.read_csv(INFILE, parse_dates=["filing_date", "period_end"])
for c in FLOW_COLS + INSTANT_COLS:
    if c in financials_09_25.columns:
        financials_09_25[c] = pd.to_numeric(financials_09_25[c], errors="coerce")

# -------- HELPERS (MSFT fiscal calendar) --------
def is_valid_period_end(d: pd.Timestamp) -> bool:
    """
    Microsoft fiscal quarters end on calendar month-ends:
      - Q1: Sep 30
      - Q2: Dec 31
      - Q3: Mar 31
      - Q4: Jun 30 (fiscal year-end)
    """
    if pd.isna(d): 
        return False
    m, day = d.month, d.day
    return (m, day) in {(9, 30), (12, 31), (3, 31), (6, 30)}

def fiscal_year_from_period_end(d: pd.Timestamp) -> int:
    """
    MSFT FY runs Jul 1 -> Jun 30.
    Periods ending Jul–Dec belong to FY (year+1); Jan–Jun belong to FY (year).
    E.g., 2024-09-30 -> FY2025; 2025-03-31 -> FY2025; 2025-06-30 -> FY2025.
    """
    if d.month >= 7:
        return d.year + 1
    return d.year

def quarter_label_from_period_end(d: pd.Timestamp) -> str:
    """
    Map period_end to MSFT fiscal quarter labels:
      - Sep 30 -> Q1 {year+1}
      - Dec 31 -> Q2 {year+1}
      - Mar 31 -> Q3 {year}
      - Jun 30 -> Q4 {year}
    """
    if   (d.month, d.day) == (9, 30):  return f"Q1 {d.year + 1}"
    elif (d.month, d.day) == (12, 31): return f"Q2 {d.year + 1}"
    elif (d.month, d.day) == (3, 31):  return f"Q3 {d.year}"
    elif (d.month, d.day) == (6, 30):  return f"Q4 {d.year}"
    else:                              return None

def fq_num_from_label(q: str):
    if isinstance(q, str) and q.startswith("Q"):
        try: return float(q[1])
        except: return np.nan
    return np.nan

# -------- 1) DEDUP: keep latest filing per (form, period_end) --------
financials_09_25 = (
    financials_09_25
    .sort_values(["period_end", "form", "filing_date"])
    .groupby(["period_end", "form"], as_index=False)
    .last()
)

# -------- 2) DROP obvious spurious period_end --------
financials_09_25 = financials_09_25[financials_09_25["period_end"].apply(is_valid_period_end)].copy()

# Keep only true annual 10-Ks at FY end (Jun 30)
financials_09_25 = financials_09_25[~(
    (financials_09_25["form"] == "10-K") &
    ~((financials_09_25["period_end"].dt.month == 6) & (financials_09_25["period_end"].dt.day == 30))
)].copy()

# -------- 3) ASSIGN fiscal labels --------
financials_09_25["fy"]      = financials_09_25["period_end"].apply(fiscal_year_from_period_end).astype("Int64")
financials_09_25["quarter"] = financials_09_25["period_end"].apply(quarter_label_from_period_end)
financials_09_25["fq"]      = financials_09_25["quarter"].map(fq_num_from_label)

# -------- 4) SYNTHESIZE Q4 if missing (flows from 10-K - (Q1+Q2+Q3); instants from 10-K) --------
q = financials_09_25[financials_09_25["form"] == "10-Q"].copy()
k = financials_09_25[financials_09_25["form"] == "10-K"].copy()

# Sum flows for Q1..Q3 per FY
sum_q1_3 = (q[q["fq"].isin([1.0, 2.0, 3.0])]
            .groupby("fy")[FLOW_COLS]
            .sum(min_count=1))

# Annual flows & instants per FY from 10-K
annual = (k.sort_values("period_end")
            .drop_duplicates(subset=["fy"], keep="last")
            .set_index("fy"))

annual_flows    = annual[FLOW_COLS] if set(FLOW_COLS).issubset(annual.columns) else pd.DataFrame(index=annual.index)
annual_instants = annual[INSTANT_COLS] if set(INSTANT_COLS).issubset(annual.columns) else pd.DataFrame(index=annual.index)
annual_meta     = annual[["filing_date", "period_end"]] if {"filing_date","period_end"}.issubset(annual.columns) else pd.DataFrame(index=annual.index)

# Compute Q4 flows
q4_flows = (annual_flows - sum_q1_3).dropna(how="all")

# Which FYs still lack a Q4 10-Q?
have_q4 = set(q[q["fq"] == 4.0]["fy"].dropna().astype(int).tolist())
need_q4 = [fy for fy in q4_flows.index if fy not in have_q4]

q4_rows = []
for fy in need_q4:
    p_end  = annual_meta.loc[fy, "period_end"] if "period_end" in annual_meta.columns else pd.Timestamp(f"{fy}-06-30")
    fdate  = annual_meta.loc[fy, "filing_date"] if "filing_date" in annual_meta.columns else p_end

    row = {
        "filing_date": fdate,
        "period_end":  p_end,
        "form":        "10-Q",
        "fy":          int(fy),
        "quarter":     f"Q4 {fy}",
        "fq":          4.0,
        "synthetic":   True,
        "note":        "Q4 flows = 10K - (Q1+Q2+Q3); instants from 10K",
    }
    for c in FLOW_COLS:
        row[c] = q4_flows.loc[fy, c] if c in q4_flows.columns else np.nan
    for c in INSTANT_COLS:
        row[c] = annual_instants.loc[fy, c] if c in annual_instants.columns else np.nan
    q4_rows.append(row)

q4_df = pd.DataFrame(q4_rows)

# Mark originals
financials_09_25["synthetic"] = False
financials_09_25["note"] = ""

# Combine and sort
full = pd.concat([financials_09_25, q4_df], ignore_index=True)
full["filing_date"] = pd.to_datetime(full["filing_date"])
full = full.sort_values(["fy", "period_end", "form", "filing_date"]).reset_index(drop=True)

# === 5) FILL missing quarters per FY (Q1..Q4) ===
quarterly_view = full[(full["form"] == "10-Q") | (full["synthetic"])].copy()

quarterly_view["fy"] = quarterly_view["quarter"].str.extract(r"\b(\d{4})\b").astype("Int64")
quarterly_view["qcode"] = quarterly_view["quarter"].str.extract(r"(Q[1-4])")

priority = {"10-Q": 0, "10-K": 1}
quarterly_view["form_priority"] = quarterly_view["form"].map(priority).fillna(2).astype(int)

quarterly_view = (
    quarterly_view.sort_values(["fy", "qcode", "form_priority", "filing_date"])
                  .groupby(["fy", "qcode"], as_index=False)
                  .last()
)

fy_min, fy_max = int(quarterly_view["fy"].min()), int(quarterly_view["fy"].max())
full_index = pd.MultiIndex.from_product(
    [range(fy_min, fy_max + 1), ["Q1", "Q2", "Q3", "Q4"]],
    names=["fy", "qcode"]
)

quarterly_full = (
    quarterly_view.set_index(["fy", "qcode"])
                  .reindex(full_index)
                  .reset_index()
)

quarterly_full["quarter"] = quarterly_full["qcode"] + " " + quarterly_full["fy"].astype("Int64").astype(str)

cols_to_keep = full.columns.tolist()
for c in cols_to_keep:
    if c not in quarterly_full.columns:
        quarterly_full[c] = np.nan

full = quarterly_full[cols_to_keep]

# Quick check
check = (full.assign(fq_lbl=lambda d: d["quarter"].str.extract(r"(Q[1-4])"))
              .pivot_table(index="fy", columns="fq_lbl", values="period_end", aggfunc="count")
              .fillna(0).astype(int))
print("Quarters present per FY:\n", check)

# Drop unneeded cols and save
full = full.drop(columns=["filing_date", "fy", "fq"])
full.to_csv(OUTFILE, index=False)
print(f"[DONE] Saved cleaned dataset → {OUTFILE}")


Quarters present per FY:
 fq_lbl  Q1  Q2  Q3  Q4
fy                    
2010     1   1   1   1
2011     1   1   1   1
2012     1   1   1   1
2013     1   1   1   1
2014     1   1   1   1
2015     1   1   1   1
2016     1   1   1   1
2017     1   1   1   1
2018     1   1   1   1
2019     1   1   1   1
2020     1   1   1   1
2021     1   1   1   1
2022     1   1   1   1
2023     1   1   1   1
2024     1   1   1   1
2025     1   1   1   1
[DONE] Saved cleaned dataset → merged_microsoft_financials_clean_09_25_missing.csv


CODE BELOW FOR FISCAL YEAR STARTING ON 01/01

In [1]:
import pandas as pd
import numpy as np

# -------- CONFIG --------
INFILE  = "/Users/jon/Desktop/Ironhack/Unit 7 - Final Project/Final-Project-Ironhack/Type of analysis/Analisis temporal/Merged files/csv/tesla/merged_tesla_financials_2009_2025.csv"
OUTFILE = "merged_tesla_financials_clean_09_25_missing.csv"

FLOW_COLS    = ["revenue", "net_income"]                  # period flows
INSTANT_COLS = ["total_assets", "total_liabilities", "shareholders_equity"]  # end-of-period

# -------- LOAD --------
financials_09_25 = pd.read_csv(INFILE, parse_dates=["filing_date", "period_end"])
for c in FLOW_COLS + INSTANT_COLS:
    if c in financials_09_25.columns:
        financials_09_25[c] = pd.to_numeric(financials_09_25[c], errors="coerce")

# -------- HELPERS --------
def is_valid_period_end(d: pd.Timestamp) -> bool:
    """
    Tesla uses calendar quarter ends:
      - Q1: Mar 31
      - Q2: Jun 30
      - Q3: Sep 30
      - Q4: Dec 31 (fiscal year-end)
    """
    if pd.isna(d): return False
    m, day = d.month, d.day
    return (m, day) in {(3, 31), (6, 30), (9, 30), (12, 31)}  # <<< CHANGED (TESLA)

def fiscal_year_from_period_end(d: pd.Timestamp) -> int:
    # For Tesla (Dec 31 year-end), fiscal year equals calendar year
    return d.year  # <<< CHANGED (TESLA)

def quarter_label_from_period_end(d: pd.Timestamp) -> str:
    # Tesla fiscal (calendar quarters):
    # Q1 ends Mar 31, Q2 Jun 30, Q3 Sep 30, Q4 Dec 31, all labeled with d.year
    if   (d.month, d.day) == (3, 31): return f"Q1 {d.year}"  # <<< CHANGED (TESLA)
    elif (d.month, d.day) == (6, 30): return f"Q2 {d.year}"  # <<< CHANGED (TESLA)
    elif (d.month, d.day) == (9, 30): return f"Q3 {d.year}"  # <<< CHANGED (TESLA)
    elif (d.month, d.day) == (12, 31): return f"Q4 {d.year}" # <<< CHANGED (TESLA)
    return None

def fq_num_from_label(q: str):
    if isinstance(q, str) and q.startswith("Q"):
        try: return float(q[1])
        except: return np.nan
    return np.nan

# -------- 1) DEDUP: keep latest filing per (form, period_end) --------
financials_09_25 = (
    financials_09_25
    .sort_values(["period_end", "form", "filing_date"])     # earliest .. latest
    .groupby(["period_end", "form"], as_index=False)
    .last()                                                 # keep latest filing
)

# -------- 2) DROP obvious spurious period_end --------
financials_09_25 = financials_09_25[financials_09_25["period_end"].apply(is_valid_period_end)].copy()

# Keep only true annual 10-Ks at FY end (Dec 31 for Tesla)
financials_09_25 = financials_09_25[~(
    (financials_09_25["form"] == "10-K") &
    ~((financials_09_25["period_end"].dt.month == 12) & (financials_09_25["period_end"].dt.day == 31))
)].copy()  # <<< CHANGED (TESLA)

# -------- 3) ASSIGN fiscal labels --------
financials_09_25["fy"]      = financials_09_25["period_end"].apply(fiscal_year_from_period_end).astype("Int64")
financials_09_25["quarter"] = financials_09_25["period_end"].apply(quarter_label_from_period_end)
financials_09_25["fq"]      = financials_09_25["quarter"].map(fq_num_from_label)

# -------- 4) SYNTHESIZE Q4 if missing (flows from 10-K - (Q1+Q2+Q3); instants from 10-K) --------
q = financials_09_25[financials_09_25["form"] == "10-Q"].copy()
k = financials_09_25[financials_09_25["form"] == "10-K"].copy()

# Sum flows for Q1..Q3 per FY
sum_q1_3 = (q[q["fq"].isin([1.0, 2.0, 3.0])]
            .groupby("fy")[FLOW_COLS]
            .sum(min_count=1))

# Annual flows & instants per FY from 10-K (one per FY after dedup)
annual = (k.sort_values("period_end")
            .drop_duplicates(subset=["fy"], keep="last")
            .set_index("fy"))

annual_flows    = annual[FLOW_COLS] if set(FLOW_COLS).issubset(annual.columns) else pd.DataFrame(index=annual.index)
annual_instants = annual[INSTANT_COLS] if set(INSTANT_COLS).issubset(annual.columns) else pd.DataFrame(index=annual.index)
annual_meta     = annual[["filing_date", "period_end"]] if {"filing_date","period_end"}.issubset(annual.columns) else pd.DataFrame(index=annual.index)

# Compute Q4 flows
q4_flows = (annual_flows - sum_q1_3).dropna(how="all")

# Which FYs still lack a Q4 10-Q?
have_q4 = set(q[q["fq"] == 4.0]["fy"].dropna().astype(int).tolist())
need_q4 = [fy for fy in q4_flows.index if fy not in have_q4]

q4_rows = []
for fy in need_q4:
    p_end  = annual_meta.loc[fy, "period_end"] if "period_end" in annual_meta.columns else pd.Timestamp(f"{fy}-12-31")  # <<< CHANGED (TESLA)
    fdate  = annual_meta.loc[fy, "filing_date"] if "filing_date" in annual_meta.columns else p_end

    row = {
        "filing_date": fdate,
        "period_end":  p_end,
        "form":        "10-Q",
        "fy":          int(fy),
        "quarter":     f"Q4 {fy}",
        "fq":          4.0,
        "synthetic":   True,
        "note":        "Q4 flows = 10K - (Q1+Q2+Q3); instants from 10K",
    }
    for c in FLOW_COLS:
        row[c] = q4_flows.loc[fy, c] if c in q4_flows.columns else np.nan
    for c in INSTANT_COLS:
        row[c] = annual_instants.loc[fy, c] if c in annual_instants.columns else np.nan
    q4_rows.append(row)

q4_df = pd.DataFrame(q4_rows)

# Mark originals
financials_09_25["synthetic"] = False
financials_09_25["note"] = ""

# Combine and sort
full = pd.concat([financials_09_25, q4_df], ignore_index=True)
full["filing_date"] = pd.to_datetime(full["filing_date"])
full = full.sort_values(["fy", "period_end", "form", "filing_date"]).reset_index(drop=True)

# === 5) AÑADIR FILAS EN BLANCO SI FALTAN TRIMESTRES (Q1..Q4) POR FY ===
# Trabajamos sobre vista trimestral (10-Q o sintéticos) para evitar duplicados por 10-K
quarterly_view = full[(full["form"] == "10-Q") | (full["synthetic"])].copy()

# Extrae 'FY' y el código de trimestre (Q1..Q4) a partir de 'quarter'
quarterly_view["fy"] = quarterly_view["quarter"].str.extract(r"\b(\d{4})\b").astype("Int64")
quarterly_view["qcode"] = quarterly_view["quarter"].str.extract(r"(Q[1-4])")

# Si hay más de una fila por (fy, q) (p.ej., múltiples filings),
# prioriza 10-Q frente a 10-K (por si se coló alguno), y la última filing_date
priority = {"10-Q": 0, "10-K": 1}
quarterly_view["form_priority"] = quarterly_view["form"].map(priority).fillna(2).astype(int)

quarterly_view = (
    quarterly_view.sort_values(["fy", "qcode", "form_priority", "filing_date"])
                  .groupby(["fy", "qcode"], as_index=False)
                  .last()
)

# Construye el MultiIndex completo FY x (Q1..Q4)
fy_min, fy_max = int(quarterly_view["fy"].min()), int(quarterly_view["fy"].max())
full_index = pd.MultiIndex.from_product(
    [range(fy_min, fy_max + 1), ["Q1", "Q2", "Q3", "Q4"]],
    names=["fy", "qcode"]
)

# Reindexa para introducir filas en blanco donde falten trimestres
quarterly_full = (
    quarterly_view.set_index(["fy", "qcode"])
                  .reindex(full_index)
                  .reset_index()
)

# Reconstruye la etiqueta 'quarter' (p.ej., "Q1 2014")
quarterly_full["quarter"] = quarterly_full["qcode"] + " " + quarterly_full["fy"].astype("Int64").astype(str)

# Integra de vuelta con el DataFrame 'full':
#  - Eliminamos posibles duplicados por 'quarter' en 'full'
#  - Usamos 'quarterly_full' como base trimestral definitiva (con NaNs donde faltaba)
cols_to_keep = full.columns.tolist()
# Asegura que 'quarterly_full' tenga todas las columnas (las ausentes se crean como NaN)
for c in cols_to_keep:
    if c not in quarterly_full.columns:
        quarterly_full[c] = np.nan

# Nos quedamos con las columnas en el mismo orden
full = quarterly_full[cols_to_keep]

# (Optional) Keep purely quarterly view (drop 10-Ks) once Q4 is synthesized:
# full = full[(full["form"] == "10-Q") | (full["synthetic"])].copy()

# Quick check: quarters present per FY
check = (full.assign(fq_lbl=lambda d: d["quarter"].str.extract(r"(Q[1-4])"))
              .pivot_table(index="fy", columns="fq_lbl", values="period_end", aggfunc="count")
              .fillna(0).astype(int))
print("Quarters present per FY:\n", check)

# Drop unecessary columns
full = full.drop(columns=["filing_date", "fy", "fq"])

# Save
full.to_csv(OUTFILE, index=False)
print(f"[DONE] Saved cleaned dataset → {OUTFILE}")


Quarters present per FY:
 fq_lbl  Q1  Q2  Q3  Q4
fy                    
2011     0   1   1   1
2012     1   1   1   1
2013     1   1   1   1
2014     1   1   1   1
2015     1   1   1   1
2016     1   1   1   1
2017     1   1   1   1
2018     1   1   1   1
2019     1   1   1   1
2020     1   1   1   1
2021     1   1   1   1
2022     1   1   1   1
2023     1   1   0   1
2024     0   0   0   0
2025     1   0   0   0
[DONE] Saved cleaned dataset → merged_tesla_financials_clean_09_25_missing.csv


FISCAL YEAR STARTS 01/02

In [1]:
import pandas as pd
import numpy as np

# -------- CONFIG --------
INFILE  = "/Users/jon/Desktop/Ironhack/Unit 7 - Final Project/Final-Project-Ironhack/Type of analysis/Analisis temporal/Merged files/csv/walmart/merged_microsoft_financials_2009_2025.csv"
OUTFILE = "merged_walmart_financials_clean_09_25_missing.csv"

FLOW_COLS    = ["revenue", "net_income"]                  # period flows
INSTANT_COLS = ["total_assets", "total_liabilities", "shareholders_equity"]  # end-of-period

# -------- LOAD --------
financials_09_25 = pd.read_csv(INFILE, parse_dates=["filing_date", "period_end"])
for c in FLOW_COLS + INSTANT_COLS:
    if c in financials_09_25.columns:
        financials_09_25[c] = pd.to_numeric(financials_09_25[c], errors="coerce")

# -------- HELPERS --------
def is_valid_period_end(d: pd.Timestamp) -> bool:
    """
    Walmart fiscal quarters end on calendar month-ends:
      - Q1: Apr 30
      - Q2: Jul 31
      - Q3: Oct 31
      - Q4: Jan 31 (fiscal year-end)
    """
    if pd.isna(d): return False
    m, day = d.month, d.day
    return (m, day) in {(4, 30), (7, 31), (10, 31), (1, 31)}  # <<< CHANGED (WMT)

def fiscal_year_from_period_end(d: pd.Timestamp) -> int:
    # For Walmart (Jan 31 year-end):
    # - Jan 31 -> FY = d.year
    # - Apr/Jul/Oct -> FY = d.year + 1
    if d.month == 1:
        return d.year                                     # <<< CHANGED (WMT)
    return d.year + 1                                     # <<< CHANGED (WMT)

def quarter_label_from_period_end(d: pd.Timestamp) -> str:
    # Walmart fiscal quarter labels:
    # - Apr 30  -> Q1 {d.year + 1}
    # - Jul 31  -> Q2 {d.year + 1}
    # - Oct 31  -> Q3 {d.year + 1}
    # - Jan 31  -> Q4 {d.year}
    if   (d.month, d.day) == (4, 30):  return f"Q1 {d.year + 1}"  # <<< CHANGED (WMT)
    elif (d.month, d.day) == (7, 31):  return f"Q2 {d.year + 1}"  # <<< CHANGED (WMT)
    elif (d.month, d.day) == (10, 31): return f"Q3 {d.year + 1}"  # <<< CHANGED (WMT)
    elif (d.month, d.day) == (1, 31):  return f"Q4 {d.year}"      # <<< CHANGED (WMT)
    return None

def fq_num_from_label(q: str):
    if isinstance(q, str) and q.startswith("Q"):
        try: return float(q[1])
        except: return np.nan
    return np.nan

# -------- 1) DEDUP: keep latest filing per (form, period_end) --------
financials_09_25 = (
    financials_09_25
    .sort_values(["period_end", "form", "filing_date"])     # earliest .. latest
    .groupby(["period_end", "form"], as_index=False)
    .last()                                                 # keep latest filing
)

# -------- 2) DROP obvious spurious period_end --------
financials_09_25 = financials_09_25[financials_09_25["period_end"].apply(is_valid_period_end)].copy()

# Keep only true annual 10-Ks at FY end (Jan 31)
financials_09_25 = financials_09_25[~(
    (financials_09_25["form"] == "10-K") &
    ~((financials_09_25["period_end"].dt.month == 1) & (financials_09_25["period_end"].dt.day == 31))
)].copy()  # <<< CHANGED (WMT)

# -------- 3) ASSIGN fiscal labels --------
financials_09_25["fy"]      = financials_09_25["period_end"].apply(fiscal_year_from_period_end).astype("Int64")
financials_09_25["quarter"] = financials_09_25["period_end"].apply(quarter_label_from_period_end)
financials_09_25["fq"]      = financials_09_25["quarter"].map(fq_num_from_label)

# -------- 4) SYNTHESIZE Q4 if missing (flows from 10-K - (Q1+Q2+Q3); instants from 10-K) --------
q = financials_09_25[financials_09_25["form"] == "10-Q"].copy()
k = financials_09_25[financials_09_25["form"] == "10-K"].copy()

# Sum flows for Q1..Q3 per FY
sum_q1_3 = (q[q["fq"].isin([1.0, 2.0, 3.0])]
            .groupby("fy")[FLOW_COLS]
            .sum(min_count=1))

# Annual flows & instants per FY from 10-K (one per FY after dedup)
annual = (k.sort_values("period_end")
            .drop_duplicates(subset=["fy"], keep="last")
            .set_index("fy"))

annual_flows    = annual[FLOW_COLS] if set(FLOW_COLS).issubset(annual.columns) else pd.DataFrame(index=annual.index)
annual_instants = annual[INSTANT_COLS] if set(INSTANT_COLS).issubset(annual.columns) else pd.DataFrame(index=annual.index)
annual_meta     = annual[["filing_date", "period_end"]] if {"filing_date","period_end"}.issubset(annual.columns) else pd.DataFrame(index=annual.index)

# Compute Q4 flows
q4_flows = (annual_flows - sum_q1_3).dropna(how="all")

# Which FYs still lack a Q4 10-Q?
have_q4 = set(q[q["fq"] == 4.0]["fy"].dropna().astype(int).tolist())
need_q4 = [fy for fy in q4_flows.index if fy not in have_q4]

q4_rows = []
for fy in need_q4:
    p_end  = annual_meta.loc[fy, "period_end"] if "period_end" in annual_meta.columns else pd.Timestamp(f"{fy}-01-31")  # <<< CHANGED (WMT)
    fdate  = annual_meta.loc[fy, "filing_date"] if "filing_date" in annual_meta.columns else p_end

    row = {
        "filing_date": fdate,
        "period_end":  p_end,
        "form":        "10-Q",
        "fy":          int(fy),
        "quarter":     f"Q4 {fy}",
        "fq":          4.0,
        "synthetic":   True,
        "note":        "Q4 flows = 10K - (Q1+Q2+Q3); instants from 10K",
    }
    for c in FLOW_COLS:
        row[c] = q4_flows.loc[fy, c] if c in q4_flows.columns else np.nan
    for c in INSTANT_COLS:
        row[c] = annual_instants.loc[fy, c] if c in annual_instants.columns else np.nan
    q4_rows.append(row)

q4_df = pd.DataFrame(q4_rows)

# Mark originals
financials_09_25["synthetic"] = False
financials_09_25["note"] = ""

# Combine and sort
full = pd.concat([financials_09_25, q4_df], ignore_index=True)
full["filing_date"] = pd.to_datetime(full["filing_date"])
full = full.sort_values(["fy", "period_end", "form", "filing_date"]).reset_index(drop=True)

# === 5) AÑADIR FILAS EN BLANCO SI FALTAN TRIMESTRES (Q1..Q4) POR FY ===
# Trabajamos sobre vista trimestral (10-Q o sintéticos) para evitar duplicados por 10-K
quarterly_view = full[(full["form"] == "10-Q") | (full["synthetic"])].copy()

# Extrae 'FY' y el código de trimestre (Q1..Q4) a partir de 'quarter'
quarterly_view["fy"] = quarterly_view["quarter"].str.extract(r"\b(\d{4})\b").astype("Int64")
quarterly_view["qcode"] = quarterly_view["quarter"].str.extract(r"(Q[1-4])")

# Si hay más de una fila por (fy, q) (p.ej., múltiples filings),
# prioriza 10-Q frente a 10-K (por si se coló alguno), y la última filing_date
priority = {"10-Q": 0, "10-K": 1}
quarterly_view["form_priority"] = quarterly_view["form"].map(priority).fillna(2).astype(int)

quarterly_view = (
    quarterly_view.sort_values(["fy", "qcode", "form_priority", "filing_date"])
                  .groupby(["fy", "qcode"], as_index=False)
                  .last()
)

# Construye el MultiIndex completo FY x (Q1..Q4)
fy_min, fy_max = int(quarterly_view["fy"].min()), int(quarterly_view["fy"].max())
full_index = pd.MultiIndex.from_product(
    [range(fy_min, fy_max + 1), ["Q1", "Q2", "Q3", "Q4"]],
    names=["fy", "qcode"]
)

# Reindexa para introducir filas en blanco donde falten trimestres
quarterly_full = (
    quarterly_view.set_index(["fy", "qcode"])
                  .reindex(full_index)
                  .reset_index()
)

# Reconstruye la etiqueta 'quarter' (p.ej., "Q1 2014")
quarterly_full["quarter"] = quarterly_full["qcode"] + " " + quarterly_full["fy"].astype("Int64").astype(str)

# Integra de vuelta con el DataFrame 'full':
#  - Eliminamos posibles duplicados por 'quarter' en 'full'
#  - Usamos 'quarterly_full' como base trimestral definitiva (con NaNs donde faltaba)
cols_to_keep = full.columns.tolist()
# Asegura que 'quarterly_full' tenga todas las columnas (las ausentes se crean como NaN)
for c in cols_to_keep:
    if c not in quarterly_full.columns:
        quarterly_full[c] = np.nan

# Nos quedamos con las columnas en el mismo orden
full = quarterly_full[cols_to_keep]

# (Optional) Keep purely quarterly view (drop 10-Ks) once Q4 is synthesized:
# full = full[(full["form"] == "10-Q") | (full["synthetic"])].copy()

# Quick check: quarters present per FY
check = (full.assign(fq_lbl=lambda d: d["quarter"].str.extract(r"(Q[1-4])"))
              .pivot_table(index="fy", columns="fq_lbl", values="period_end", aggfunc="count")
              .fillna(0).astype(int))
print("Quarters present per FY:\n", check)

# Drop unecessary columns
full = full.drop(columns=["filing_date", "fy", "fq"])

# Save
full.to_csv(OUTFILE, index=False)
print(f"[DONE] Saved cleaned dataset → {OUTFILE}")


Quarters present per FY:
 fq_lbl  Q1  Q2  Q3  Q4
fy                    
2010     0   1   1   1
2011     1   1   1   1
2012     1   1   1   1
2013     1   1   1   1
2014     1   1   1   1
2015     1   1   1   1
2016     1   1   1   1
2017     1   1   1   1
2018     1   1   1   1
2019     1   1   1   1
2020     1   1   1   1
2021     1   1   1   1
2022     1   1   1   1
2023     1   1   1   1
2024     1   1   1   1
2025     1   1   1   1
2026     1   0   0   0
[DONE] Saved cleaned dataset → merged_walmart_financials_clean_09_25_missing.csv
